# Repository edition: 07_reviewer_followups

Run from this repository in your existing GPU environment. Source is in `../suites/07_reviewer_followups/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Reviewer follow-ups: eight experiments
All code is embedded. Uses your existing environment and Hugging Face access: **no installation, venv, kernel setup, or token/login cells**.

Compatible previous raw results are reused when present. If you no longer have them, the suite builds a fresh PopQA cohort. Summary ZIPs alone do not contain gates, attention rows, or first-token vectors; this suite captures those quantities and saves compact sufficient statistics.

Defaults launch the core profiles/regime/variance/sink pass, including Qwen2.5 3B and OLMo2 7B. Nulls, mechanism attribution, gradient certificates, and the source-layer matrix are selectable subsequent stages. Read the workload plan before launching. No pretrained GPU runtime has been benchmarked.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, time, zipfile
BUNDLE_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('07_reviewer_followups'))).hexdigest()
BUNDLE = _source_bundle('07_reviewer_followups')

payload = base64.b64decode(BUNDLE)
assert hashlib.sha256(payload).hexdigest() == BUNDLE_SHA256
CODE = Path.home() / ("threeways_reviewer_code_" + BUNDLE_SHA256[:12])
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        if CODE.resolve() not in (CODE / item.filename).resolve().parents:
            raise ValueError("Unsafe archive path")
    archive.extractall(CODE)

def run_command(args):
    result = subprocess.run(args, cwd=CODE, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    return result

print("Code ready:", CODE)


## 1. Existing environment and optional previous results
The previous directory is read-only. If it is absent, fresh data will be prepared automatically.


In [ ]:
candidates = [Path("/home/ubuntu/venvs/threeways/bin/python"), Path(sys.executable)]
candidates += [Path.home() / name / "bin/python" for name in ["threeways_followup_env", "threeways_causal_env"]]
check = "import torch,transformers,numpy,pandas,matplotlib,huggingface_hub; assert transformers.__version__=='4.48.3', 'Expected tested Transformers 4.48.3'; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name())"
PYTHON = None
errors = []
for candidate in dict.fromkeys(candidates):
    if not candidate.exists(): continue
    result = subprocess.run([str(candidate), "-c", check], capture_output=True, text=True)
    if result.returncode == 0:
        PYTHON = candidate
        print("Using:", candidate, "|", result.stdout.strip())
        break
    errors.append(str(candidate) + ": " + result.stderr[-1000:])
if PYTHON is None:
    raise RuntimeError("Point candidates to your existing working environment.\n" + "\n".join(errors))

PREVIOUS = Path.home() / "threeways_channel_results" / "channels_fresh_v1"
if not PREVIOUS.exists():
    PREVIOUS = None
    print("Previous results not found: a fresh cohort will be created.")
else:
    print("Optional previous results:", PREVIOUS)

RESULTS = Path.home() / "threeways_channel_followups" / "reviewer_v1"
RESULTS.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RESULTS / "configuration.json"
LOG = RESULTS / "execution.log"


## 2. Configure the campaign and print workload
The core pass covers the full evaluation cohort on all six models. The more expensive experiments use explicit, prespecified subsets:

- Nulls: 100 matched prompts on 1B; one random initialization and three random-readout seeds.
- Three-mechanism attribution: 100 prompts on 1B and 8B; eight backward passes per prompt.
- Gradient certificates: 16 prompts/model, three depths, all projection families, random and gradient-aligned directions, three epsilon values.
- Source × reselecting layer matrix: 100 prompts on 1B and 8B; `L+2` backward passes per prompt.

Set the corresponding `*_prompts` to `None` **before the first launch** to use every evaluation prompt. Three singleton freezes are also reported, but their effects overlap; the Shapley attribution supplies the additive decomposition.

The regime definitions are new, explicit, vocabulary-centered origin diagnostics calibrated on independent successes. They retain continuation, mixed, and uncalibrated cases rather than forcing every error into three categories. See README.txt in CODE for the exact definitions.


In [ ]:
CONFIG = json.loads((CODE / "followup_study/config.json").read_text())

# Initial core pass: experiments 4, 5, 7, 8.
STAGES = ["profiles", "regimes", "variance", "sink"]

# Later stages, using the SAME configuration and RESULTS directory:
# STAGES = ["nulls"]       # experiment 1
# STAGES = ["certificate"] # experiment 3
# STAGES = ["freeze"]      # experiment 2
# STAGES = ["matrix"]      # experiment 6
# Or request all together (potentially long):
# STAGES = ["profiles", "regimes", "variance", "sink", "nulls", "certificate", "freeze", "matrix"]

# Optional BEFORE the first launch:
# CONFIG["freeze_prompts"] = None
# CONFIG["certificate_prompts"] = None
# CONFIG["null_prompts"] = None
# CONFIG["cpu_threads"] = 8

if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != CONFIG:
    raise RuntimeError("Configuration differs. Use a new RESULTS folder for a different protocol.")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
previous_args = ["--previous", str(PREVIOUS)] if PREVIOUS is not None else []
run_command([str(PYTHON), "-m", "followup_study.plan", "--config", str(CONFIG_PATH),
             "--out", str(RESULTS), "--stages", *STAGES, *previous_args])


## 3. Launch a detached worker
Closing the browser or interrupting a monitor cell does not stop this worker. The GPU instance must remain powered on. All completed records are saved incrementally. Previous results are not modified.


In [ ]:
import fcntl
for lock_name in ["run.lock", "supervisor.lock"]:
    with (RESULTS / lock_name).open("a+") as handle:
        try:
            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError("A worker/supervisor is already running. Use the monitor.")
if "supervisor" in globals() and supervisor.poll() is None:
    raise RuntimeError("This notebook already launched a supervisor. Use the monitor.")
if (RESULTS / "STOP").exists():
    raise RuntimeError("STOP exists. Use the resume instructions below first.")
(RESULTS / "exit.json").unlink(missing_ok=True)
command = [str(PYTHON), "-u", "-m", "followup_study.supervisor", "--config", str(CONFIG_PATH),
           "--out", str(RESULTS), "--stages", *STAGES, *previous_args]
with (RESULTS / "supervisor.log").open("a") as f:
    supervisor = subprocess.Popen(command, cwd=CODE, stdout=f, stderr=subprocess.STDOUT,
                                  start_new_session=True)
print("Supervisor PID:", supervisor.pid)
print("Monitor:", LOG)


## 4. Monitor
Rerun this cell to see progress. A zero exit code with `stopped: true` means a requested stop, not completion.


In [ ]:
if LOG.exists():
    with LOG.open("rb") as f:
        f.seek(max(0, LOG.stat().st_size - 14000))
        print(f.read().decode(errors="replace"))
else:
    print("Waiting for log.")
if (RESULTS / "exit.json").exists():
    print("Worker exit:", json.loads((RESULTS / "exit.json").read_text()))


## 5. Refresh and print compact reports
Use after a stage finishes, or for a partial descriptive preview. This reads completed records and needs no GPU. Partial calibration and partial cohorts are not final estimates. Regime joins do not modify your previous run.


In [ ]:
run_command([str(PYTHON), "-m", "followup_study.run", "--config", str(CONFIG_PATH),
             "--out", str(RESULTS), "--stages", "report"])
import csv
for name in ["regime_counts.csv", "calibration.csv", "null_paired_summary.csv", "freeze_summary.csv", "certificate_summary.csv"]:
    path = RESULTS / "reports" / name
    if not path.exists(): continue
    print("\n" + name)
    with path.open() as f:
        for i, row in enumerate(csv.DictReader(f)):
            if i == 6: break
            keep = {k:v for k,v in row.items() if not k.endswith(("_lo", "_hi", "_valid_facts"))}
            print(keep)
print("All tables and figures:", RESULTS / "reports")


## 6. Stop and resume
Run the stop request only when you want to stop. Wait for `exit.json` in the monitor before terminating the instance or clearing STOP. The current operation may finish first.

To launch the next experiment, change **only STAGES** in cell 2, rerun that cell, then launch again. Completed work is reused. Keep configuration and cohort unchanged.


In [ ]:
# STOP (uncomment to request):
# (RESULTS / "STOP").touch()
# print("Stop requested; monitor until exit.json appears.")

# RESUME (uncomment only after the worker has exited):
# for lock_name in ["run.lock", "supervisor.lock"]:
#     with (RESULTS / lock_name).open("a+") as handle:
#         fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
# (RESULTS / "STOP").unlink(missing_ok=True)
# print("Ready. Rerun the launcher with the desired STAGES.")


## 7. Small, handy export for review
This exports summary tables, figures, configuration, provenance, and calibrated label mappings. **It excludes raw per-prompt measurements and statistic states**, so it is not a complete resume backup. Download before terminating the instance. Keep a full copy separately if you want arbitrary later reanalysis.


In [ ]:
from datetime import datetime
from IPython.display import display, FileLink
ARCHIVE = RESULTS.parent / (RESULTS.name + "_summaries_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip")
files = set()
for pattern in ["reports/*.csv", "reports/*.pdf", "reports/*.json", "*/model.json", "*/architecture.json", "*/calibration.json", "*/certificate_status.json"]:
    files.update(RESULTS.glob(pattern))
for name in ["configuration.json", "protocol.json", "environment.json", "completion.json", "exit.json", "data/cohort.json"]:
    if (RESULTS / name).exists(): files.add(RESULTS / name)
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(files): archive.write(path, path.relative_to(RESULTS))
with zipfile.ZipFile(ARCHIVE) as archive:
    assert archive.testzip() is None
print(f"Verified: {len(files)} files, {ARCHIVE.stat().st_size / 2**20:.2f} MiB")
print(ARCHIVE)
display(FileLink(os.path.relpath(ARCHIVE, Path.cwd())))


## Optional: local numerical tests
No model downloads; tests use tiny random CPU models. They verify gradients, freeze coalitions, Shapley closure, origin invariance, streamed statistics, diagnostic branches and pipeline resumption. Passing these is not a pretrained GH200 benchmark.


In [ ]:
# Optional:
# run_command([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-v"])
